### Prerequisites

```bash
git clone https://github.com/Znbne/telco-retrieve
cd open-telco-rag
pip install -e .
```

Set your API keys in a `.env` file at the project root:
```
# For answer generation (choose one or both)
ANTHROPIC_API_KEY=sk-ant-...        # Claude models direct
OPENROUTER_API_KEY=sk-or-...        # Any model via OpenRouter
```


## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Project paths
DATA_DIR   = Path("data/oran")
DB_PATH    = "chroma_db"
MODEL_KEY  = "minilm"        # all-MiniLM-L6-v2 — fast development baseline
COLLECTION = f"oran_{MODEL_KEY}"

print(f"Data directory : {DATA_DIR}")
print(f"ChromaDB path  : {DB_PATH}")
print(f"Embedding model: {MODEL_KEY}")
print(f"Collection     : {COLLECTION}")

## 2. Download O-RAN Specifications

O-RAN Alliance specifications are publicly available at
[orandownloadsweb.azurewebsites.net](https://orandownloadsweb.azurewebsites.net/specifications).
Download them manually and place in `data/oran/`, or use the GSMA download script:

```bash
python scripts/download_oran.py --output data/oran
```

The cell below checks how many files are available.

In [ ]:
# Count available O-RAN files
extensions = [".pdf", ".docx"]
files = [f for f in DATA_DIR.rglob("*") if f.suffix.lower() in extensions]

from collections import Counter
by_ext = Counter(f.suffix.lower() for f in files)

print(f"Total O-RAN files: {len(files)}")
for ext, count in sorted(by_ext.items()):
    print(f"  {ext}: {count}")

# Sample filenames
print("\nSample files:")
for f in sorted(files)[:5]:
    print(f"  {f.name}")

## 3. Ingest into ChromaDB

The ingest pipeline:
1. Loads each file using `ORANSource` (PDF/DOCX → text)
2. Applies O-RAN-specific cleaning (removes page numbers, headers, ToC entries)
3. Chunks the text at 1,024-character boundaries with 200-character overlap
4. Embeds each chunk with `all-MiniLM-L6-v2`
5. Stores vectors + metadata in ChromaDB

**Resume support**: re-running this cell skips already-ingested files.
Use `resume=False` to force a full re-ingest.

In [ ]:
from open_telco_rag.sources.oran import ORANSource
from open_telco_rag.embedders.sentense_transformers import SentenceTransformerEmbedder
from open_telco_rag.stores.chroma import ChromaStore
from open_telco_rag.core.pipeline import ingest

source   = ORANSource()
embedder = SentenceTransformerEmbedder(model_key=MODEL_KEY)
store    = ChromaStore(path=DB_PATH, collection_name=COLLECTION)

print(f"Starting ingest — {store.count()} chunks already in collection")

stats = ingest(
    source     = source,
    store      = store,
    embedder   = embedder,
    data_dir   = str(DATA_DIR),
    chunk_size = 1024,
    overlap    = 200,
    resume     = True,   # set False to force full re-ingest
)

print(f"\nIngest complete")
print(f"  Files processed : {stats.files_processed}")
print(f"  Chunks stored   : {stats.chunks_stored}")
print(f"  Skipped         : {stats.skipped_files}")
print(f"  Errors          : {stats.error_files}")
print(f"  Total in DB     : {store.count():,}")

## 4. Explore the Collection

Before querying, inspect what's in the collection — chunk counts, work group
distribution, and chunk length statistics.

In [ ]:
import chromadb
from collections import Counter

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(COLLECTION)

# Sample 1000 chunks to profile the collection
sample = collection.get(limit=1000, include=["documents", "metadatas"])

# Work group distribution
wg_counts = Counter(
    m.get("work_group", "unknown") for m in sample["metadatas"]
)

# Chunk length distribution
lengths = [len(d) for d in sample["documents"]]

print(f"Collection: {COLLECTION}")
print(f"Total chunks: {collection.count():,}\n")

print("Work group distribution (sample of 1,000 chunks):")
print(f"{'Work Group':<12} {'Chunks':>8}")
print("-" * 22)
for wg, count in sorted(wg_counts.items(), key=lambda x: -x[1]):
    print(f"{wg:<12} {count:>8}")

import statistics
print(f"\nChunk length statistics (chars):")
print(f"  Mean   : {statistics.mean(lengths):.0f}")
print(f"  Median : {statistics.median(lengths):.0f}")
print(f"  Min    : {min(lengths)}")
print(f"  Max    : {max(lengths)}")

## 5. Semantic Search

Run a semantic search query and inspect the results — similarity scores,
source documents, and work groups.

In [ ]:
from open_telco_rag.core.pipeline import search

QUERY = "What is the E2 interface and how do xApps use it?"
TOP_K = 5

results = search(
    QUERY,
    store    = store,
    embedder = embedder,
    top_k    = TOP_K,
)

print(f"Query: '{QUERY}'")
print(f"Top-{TOP_K} results:\n")
print(f"{'Rank':<5} {'Similarity':>10} {'WG':<6} {'Source'}")
print("-" * 70)

for i, r in enumerate(results, 1):
    sim    = round(1 - r.distance, 3)
    source = r.metadata.get("source", "")[:45]
    wg     = r.metadata.get("work_group", "")
    print(f"{i:<5} {sim:>10.3f} {wg:<6} {source}")

print(f"\nTop result preview:")
print(results[0].text[:400])

### 5.1 Effect of top-k on retrieval quality



In [ ]:
QUERY = "How does O-RAN support network slicing?"

print(f"Query: '{QUERY}'\n")
print(f"{'top-k':<8} {'Unique WGs':<12} {'Mean sim':>10} {'WGs retrieved'}")
print("-" * 65)

for k in [3, 5, 8, 10]:
    results = search(QUERY, store=store, embedder=embedder, top_k=k)
    sims    = [1 - r.distance for r in results]
    wgs     = list({r.metadata.get("work_group", "?") for r in results})
    mean_s  = sum(sims) / len(sims) if sims else 0
    print(f"{k:<8} {len(wgs):<12} {mean_s:>10.3f} {', '.join(sorted(wgs))}")

## 6. Question Answering

Ask a natural language question and receive a grounded answer from the
O-RAN specifications. The answer cites specific source documents.

In [ ]:
from open_telco_rag.core.pipeline import generate
from open_telco_rag.evaluation._claude import make_llm_call

QUESTION = "What is the role of the Near-RT RIC in O-RAN architecture?"

retrieved = search(QUESTION, store=store, embedder=embedder, top_k=5)

# Use OpenRouter for generation — set your preferred model here
from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

response  = generate(QUESTION, retrieved, source_type="oran", generator=generator)

print(f"Question: {QUESTION}")
print(f"{'─'*70}")
print(f"\n{response['answer']}")
print(f"\n{'─'*70}")
print(f"Sources used: {response['sources']}")
print(f"Chunks used : {response['chunks_used']}")


### 6.1 Question bank

Try the full O-RAN evaluation question set across all work groups.

In [ ]:
QUESTIONS = [
    ("What is the role of the Near-RT RIC in O-RAN architecture?",    "WG3"),
    ("How does O-RAN support network slicing?",                        "WG1"),
    ("What is the A1 interface used for?",                             "WG2"),
    ("What are the main use cases in the O-RAN use cases report?",     "WG1"),
    ("How does the SMO interact with other O-RAN components?",         "WG1"),
    ("What are the security requirements for xApps in O-RAN?",         "WG11"),
    ("What is the O2 interface and what does it enable?",              "WG6"),
    ("How does O-RAN define the Open Fronthaul interface?",            "WG4"),
    ("What AI/ML use cases does O-RAN define for the Non-RT RIC?",     "WG2"),
    ("What is the E2 service model and how do xApps use it?",          "WG3"),
]

print(f"{'#':<3} {'Expected WG':<14} {'Top WG retrieved':<18} {'Mean sim':>9} {'Question'}")
print("-" * 90)

for i, (question, expected_wg) in enumerate(QUESTIONS, 1):
    results  = search(question, store=store, embedder=embedder, top_k=5)
    sims     = [1 - r.distance for r in results]
    mean_sim = sum(sims) / len(sims) if sims else 0
    
    # Most frequent WG in top-5
    wg_counts = Counter(r.metadata.get("work_group", "?") for r in results)
    top_wg    = wg_counts.most_common(1)[0][0] if wg_counts else "?"
    hit       = "✓" if top_wg == expected_wg else "✗"
    
    q_short = question[:45] + "..." if len(question) > 45 else question
    print(f"{i:<3} {expected_wg:<14} {top_wg + ' ' + hit:<18} {mean_sim:>9.3f} {q_short}")


## 7. Evaluation

Run the full evaluation suite across all O-RAN evaluation questions.
This uses the evaluation framework from `open_telco_rag/evaluation/`.

**Note**: metric scoring calls the LLM API (configured below).
The full run takes approximately 3–5 minutes for 10 questions.

In [ ]:
from open_telco_rag.evaluation.evaluate import evaluate_corpus
from open_telco_rag.evaluation.questions import load_questions

questions = load_questions("oran")
print(f"Loaded {len(questions)} evaluation questions for O-RAN")
print(f"Questions with reference answers: "
      f"{sum(1 for q in questions if q.reference_answer)}")

In [ ]:
from open_telco_rag.evaluation._claude import make_llm_call

# RAG generation model (answers questions)
rag_llm_call = make_llm_call("openrouter", "meta-llama/llama-3.3-70b-instruct")

# Metrics evaluation model (scores faithfulness, relevancy, etc.)
llm_call = make_llm_call("openrouter", "anthropic/claude-sonnet-4.5")

result = evaluate_corpus(
    corpus       = "oran",
    store        = store,
    embedder     = embedder,
    questions    = questions,
    model        = MODEL_KEY,
    top_k        = 5,
    delay        = 1.0,
    llm_call     = llm_call,
    rag_llm_call = rag_llm_call,
)


In [ ]:
# Per-question results table
print(f"\nPer-question results — O-RAN ({MODEL_KEY}, top-k=5)")
print(f"{'#':<3} {'Faith':>6} {'Relev':>6} {'Sim':>6} {'Recall':>7} {'Correct':>8} {'Question'}")
print("-" * 95)

for i, qr in enumerate(result.question_results, 1):
    q_short = qr.question[:45] + "..." if len(qr.question) > 45 else qr.question
    print(
        f"{i:<3} "
        f"{qr.faithfulness:>6.2f} "
        f"{qr.answer_relevancy:>6.2f} "
        f"{qr.mean_similarity:>6.3f} "
        f"{max(0, qr.context_recall):>7.2f} "
        f"{max(0, qr.answer_correctness):>8.2f} "
        f"{q_short}"
    )

print("-" * 95)
print(
    f"{'AVG':<3} "
    f"{result.mean_faithfulness:>6.2f} "
    f"{result.mean_answer_relevancy:>6.2f} "
    f"{result.mean_similarity:>6.3f} "
    f"{max(0, result.mean_context_recall):>7.2f} "
    f"{max(0, result.mean_answer_correctness):>8.2f}"
)


## 8. What the metrics tell us

| Metric | Type | Interpretation |
|--------|------|----------------|
| **Faithfulness** | Generation | Are the claims in the answer grounded in the retrieved chunks? |
| **Answer relevancy** | Generation | Does the answer address what was asked? |
| **Mean similarity** | Retrieval | How confident is the retrieval? Higher = better chunk matches |
| **Context recall** | Retrieval+GT | Were the chunks needed to answer the question retrieved? |
| **Answer correctness** | Generation+GT | Is the answer factually correct vs the reference answer? |
